Feature engineering depends on what type of ML model you will use. For our project, we will use collaborative filtering. In this method, we will predict reccomendations based on the assumption
that the best recommendations come from people who have similar tastes. In other words, it uses historical item ratings of like-minded people to predict how someone would rate an item.


Collaborative filtering predicts the rating or preference of a user for a particular item. This rating can then be used to make recommendations for the user.

For example, if a user has rated several items highly, a collaborative filtering model can predict the user's rating for other items and recommend those with the highest predicted rating. In other words, the model recommends items based on the user's past behavior and the behavior of similar users.

To recommend items to a user, you can use the trained collaborative filtering model to predict the rating of the user for all the items in the dataset. Then, you can recommend the items with the highest predicted ratings to the user.

So, in summary, collaborative filtering is used to predict the rating or preference of a user for a particular item. The predicted ratings can then be used to make recommendations for the user.

In this case, the asin_encoded and reviewerID_encoded columns represent the input features that describe the relationships between users and items in the dataset. The overall column represents the target variable that the model will try to predict, which is the rating that a user would give to a particular item.

We will also drop all other columns.

By training a collaborative filtering model on this data, you can generate recommendations for users based on their past ratings and the ratings of similar users.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType

spark=SparkSession \
    .builder \
    .appName("FT") \
    .master("local") \
    .config("spark.mongodb.input.uri", "mongodb://localhost:27017/") \
    .config("spark.mongodb.output.uri", "mongodb://localhost:27017/") \
    .config("spark.driver.memory","12g")\
    .config("spark.executor.memory", "12g") \
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:3.0.1") \
    .getOrCreate()


df = spark.read.format("com.mongodb.spark.sql.DefaultSource").option("pipeline", "[{'$sample': {'size': 100000} }]")\
    .option("uri", "mongodb://localhost:27017/"+"amazon"+"."+"data")\
    .option("partitioner", "MongoSinglePartitioner") \
    .option("partitionkey", "asin")\
    .load()

In [ ]:
df.show()

In [ ]:
df = df.drop('_id','reviewText','reviewerName','summary','unixReviewTime','verified')
df.show()

In [ ]:
from pyspark.ml.feature import StringIndexer

reviwerID_indexer= StringIndexer(inputCol="reviewerID", outputCol="reviewerID_index",handleInvalid='keep')
fin_reviwerID= reviwerID_indexer.fit(df).transform(df)
# fin_reviwerID.show()

In [ ]:
#sample_df.printSchema()
asin_indexer= StringIndexer(inputCol="asin", outputCol="asin_index",handleInvalid='keep')
fin_asi= asin_indexer.fit(df).transform(df)
#?fin_asi.show()

In [ ]:
fin_asi = fin_asi.drop("overall")

In [ ]:
from pyspark.sql.functions import col

final_df = (fin_asi
            .join(fin_reviwerID, ['reviewerID'], 'inner')
            .select(col('asin_index'), col('reviewerID_index'), col('overall'))
           )


In [ ]:
final_df.show()

In [ ]:
# Drop duplicates based on 'asin_indexed' and 'reviewerID_indexed' columns
final_df = final_df.dropDuplicates(['asin_index', 'reviewerID_index'])

# Drop NaN values
final_df = final_df.na.drop()


In [ ]:
final_df.show()

In [ ]:
#df_index = df_index.drop('asin','reviewerID')
#df_index = df_index.dropna()

In [ ]:
#df_index.show()

In [ ]:
#from pyspark.sql.functions import count
#final_df.groupBy('overall').agg(count('*').alias('count')).show()


In [ ]:
#df_maj=df_index[final_df['overall']==5.0]
#f_min = df_index[final_df['overall']!=5.0]

In [ ]:
from pyspark.sql.functions import col

#upsample_df = df_min.sample(True, float(df_maj.count())/float(df_min.count()), seed=50)
#upsample_df = upsample_df.union(df_maj)
#upsample_df.groupBy("overall").count().orderBy(col("count").desc()).show()


In [ ]:
#upsample_df.head(5)

In [ ]:
#df_index = upsample_df

In [ ]:
train_data, test_data = final_df.randomSplit([0.75,0.25])

In [ ]:
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline

print("Reg Parm worked successfully")

# Train the ALS model on the training set
als_mf = ALS(maxIter=5, regParam=0.01, userCol="reviewerID_index", itemCol="asin_index", ratingCol="overall", coldStartStrategy="drop")
pipeline_mf = Pipeline(stages=[als_mf])
model_mf = pipeline_mf.fit(train_data)

# Make predictions on the testing set
predictions = model_mf.transform(test_data)
predictions = predictions.dropna()

# Evaluate the model using RMSE
evaluator = RegressionEvaluator(metricName="rmse", labelCol="overall", predictionCol="prediction")
rmse = evaluator.evaluate(predictions)
print("Root-mean-square error = " + str(rmse))


In [ ]:
predictions.head(5)

In [ ]:
predictions.show()

In [ ]:
model.getBlockSize()

In [ ]:
df_index = df_index.drop('asin','reviewerID')
df_index.show()

In [ ]:
from pyspark.ml.feature import VectorAssembler
featureassembler = VectorAssembler(inputCols=["asin_indexed", "reviewerID_indexed"],
                                   outputCol="VectorFeature")
df_index = featureassembler.transform(df_index)
df_index.show()

In [ ]:
df = df_index.select("VectorFeature","overall")
df.show()

In [ ]:
predictions.show()